# FloodValue — leave-one-event-out (LOEO) U-Net в Colab (#86, критерии 7 и 8)

Обёртка над **одной командой** `python -m floodvalue.model.unet.loeo` (D-036: обучение, в том числе
LOEO, запускается только в Colab). Логики обучения здесь нет — окружение, данные, запуск,
выгрузка CSV/JSON фолдов.

Протокол (`docs/verification/transfer-loeo.md`): 6 фолдов по событиям train+dev (Ghana, India,
Pakistan, Paraguay, Somalia, USA); U-Net по рецепту D-019 (60 эпох, один seed) обучается на 5
событиях, отложенное оценивается на фиксированных эпохах 19 и 60 (без выбора чекпойнта);
нормализация — по обучающим событиям фолда; baseline — на тех же валидных пикселях. Validation и
test не читаются. `models/` не меняется.

- **Среда выполнения**: *Runtime → Change runtime type → GPU* (член D-019 обучался 64–70 с на GPU,
  фолд с загрузкой чипов — дольше, 6 фолдов — порядка десятков минут; на CPU — часы, для проверки проводки есть
  `MODE = "smoke"`).
- **GPU-torch** — отдельный venv `scripts/setup_gpu_env.sh` (D-024); `uv.lock` не меняется.
- **Данные** Sen1Floods11 — `python -m floodvalue data` из `gs://sen1floods11` в `FLOODVALUE_DATA_DIR`.
- **Приватный репозиторий**: секрет Colab `GITHUB_TOKEN` — только для `git clone`.

Итог — архив `loeo_<MODE>_<commit>.zip` (`loeo-metrics.csv`, `loeo-chips.csv`, `loeo-summary.json`,
`loeo.log`) в `MyDrive/floodvalue/` или скачиванием. В репозиторий его переносят в
`docs/verification/transfer-loeo*.csv|json`, таблица — `--table-from`.


In [ ]:
# --- Параметры -------------------------------------------------------------------------
import os

def param(name, default):
    return os.environ.get(f"FLOODVALUE_NB_{name}", default)

REPO_URL = param("REPO_URL", "https://github.com/Chichetin/kosmo-moskva_umstvenno_ustalye_34.git")
REF = param("REF", "main")                    # ветка, тег или commit
WORKDIR = param("WORKDIR", "/content")        # куда клонировать и где хранить данные/результаты
MODE = param("MODE", "full")                  # "full" — 6 фолдов x 60 эпох, один seed (D-019, D-036)
                                              # "smoke" — фолд USA, 2 эпохи, 4 чипа на событие: проверка проводки, числа не результат
USE_GPU = param("USE_GPU", "auto")            # "auto" | "1" | "0"
DATA_DIR = param("DATA_DIR", "")              # пусто -> <WORKDIR>/data; можно путь на Drive, чтобы не качать заново
SAVE_TO_DRIVE = param("SAVE_TO_DRIVE", "1")   # "1" — копия архива результатов в MyDrive/floodvalue (в Colab)

REPO_DIR = os.path.join(WORKDIR, "floodvalue")
DATA_DIR = DATA_DIR or os.path.join(WORKDIR, "data")
OUT_DIR = os.path.join(WORKDIR, "outputs")
LOEO_DIR = os.path.join(OUT_DIR, "loeo" if MODE == "full" else "loeo-smoke")
GPU_VENV = os.path.join(WORKDIR, "floodvalue-gpu")   # свой venv, не общий $HOME/.cache (D-024)
assert MODE in ("full", "smoke"), MODE
print(f"REPO_URL={REPO_URL} REF={REF} MODE={MODE}\nREPO_DIR={REPO_DIR}\nDATA_DIR={DATA_DIR}\nOUT_DIR={OUT_DIR}")

In [ ]:
# --- Вспомогательное: запуск команд с потоковым выводом и проверкой кода возврата ---------
import shlex, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd, cwd=None, env=None, secret=None):
    """Run a shell command, stream its output, fail the cell on a non-zero exit code."""
    shown = cmd.replace(secret, "***") if secret else cmd
    print(f"$ {shown}", flush=True)
    proc = subprocess.Popen([shutil.which("bash") or "bash", "-c", f"set -eo pipefail; {cmd}"], cwd=cwd,
                            env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line.replace(secret, "***") if secret else line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"exit code {proc.returncode}: {shown}")

print("Colab" if IN_COLAB else "не Colab (локальный прогон)")

In [ ]:
if IN_COLAB and any(p.startswith("/content/drive") for p in (DATA_DIR, WORKDIR)):
    from google.colab import drive
    drive.mount("/content/drive")

## 1. Клон репозитория

In [ ]:
token = ""
if IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:  # no secret defined / no access granted -> public clone
        token = ""
clone_url = REPO_URL
if token and REPO_URL.startswith("https://"):
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)

os.makedirs(WORKDIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Weights in models/ are Git LFS objects; training does not need them.
    sh(f"git clone {shlex.quote(clone_url)} {shlex.quote(REPO_DIR)}",
       env={"GIT_LFS_SKIP_SMUDGE": "1"}, secret=token or None)
    sh(f"git remote set-url origin {shlex.quote(REPO_URL)}", cwd=REPO_DIR)
# Always the freshly fetched REF (branch, tag or commit), never a stale local branch.
sh(f"git fetch --quiet {shlex.quote(clone_url)} {shlex.quote(REF)}", cwd=REPO_DIR,
   env={"GIT_LFS_SKIP_SMUDGE": "1"}, secret=token or None)
sh("git checkout --quiet --detach FETCH_HEAD", cwd=REPO_DIR, env={"GIT_LFS_SKIP_SMUDGE": "1"})
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("commit", COMMIT)

## 2. uv и окружение

`uv sync --frozen` — окружение строго по `uv.lock` (CPU-torch, как у жюри). При наличии GPU
поверх ставится отдельный venv с GPU-torch: `scripts/setup_gpu_env.sh` (D-024), LOEO
идёт его python. Команда обучения одна и та же.

In [ ]:
if shutil.which("uv") is None:
    sh(f"{shlex.quote(sys.executable)} -m pip install --quiet uv")
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")
sh(f"{UV} --version")
sh(f"{UV} sync --frozen", cwd=REPO_DIR)

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"], capture_output=True).returncode == 0
assert USE_GPU in ("auto", "1", "0"), USE_GPU
gpu = {"auto": has_gpu, "1": True, "0": False}[USE_GPU]
if gpu:
    sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")
    sh("bash scripts/setup_gpu_env.sh", cwd=REPO_DIR,
       env={"FLOODVALUE_GPU_VENV": GPU_VENV, "PATH": os.path.dirname(UV) + ":" + os.environ["PATH"]})
    PY = f"{shlex.quote(os.path.join(GPU_VENV, 'bin', 'python'))}"
else:
    PY = f"{UV} run --frozen python"
ENV = {"FLOODVALUE_DATA_DIR": DATA_DIR, "FLOODVALUE_OUTPUT_DIR": OUT_DIR, "FLOODVALUE_COMMIT": COMMIT}
sh(f"{PY} -c 'import torch; print(\"torch\", torch.__version__, \"cuda\", torch.cuda.is_available())'",
   cwd=REPO_DIR, env=ENV)

## 3. Sen1Floods11

`python -m floodvalue data` (`make data`): качает недостающие HandLabeled-слои S1Hand, LabelHand,
JRCWaterHand и сплиты из `gs://sen1floods11` v1.1 (≈ 0,7 ГБ, без авторизации), проверяет md5,
делает QC и пишет `interim/qc.csv`, `interim/source_manifest_s1.json`. Повторный запуск ничего не
качает. Чтобы не качать в каждой сессии, укажите `DATA_DIR` на Google Drive — ячейка перед разделом 1 смонтирует
Drive, если `DATA_DIR` или `WORKDIR` лежат в `/content/drive`.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
sh(f"{PY} -m floodvalue data", cwd=REPO_DIR, env=ENV)

## 4. LOEO

- `full`: `python -m floodvalue.model.unet.loeo --out-dir <LOEO_DIR>` — 6 фолдов, рецепт D-019
  (60 эпох), оценка на эпохах 19 и 60. CSV/JSON пишутся после каждого фолда, так что при обрыве
  сессии готовые фолды не пропадают (для надёжности задайте `WORKDIR` на Drive).
- `smoke`: тот же модуль с `--events USA --max-chips 4 --epochs 2 --eval-epochs 1,2` — проверяет,
  что окружение и данные доходят до обучения и отчёта. Эти числа не результат.


In [ ]:
import time
LOEO_STARTED = time.time()
os.makedirs(LOEO_DIR, exist_ok=True)
LOG = os.path.join(LOEO_DIR, "loeo.log")
if MODE == "full":
    args = ""
else:
    args = "--events USA --max-chips 4 --epochs 2 --eval-epochs 1,2"
sh(f"{PY} -m floodvalue.model.unet.loeo --out-dir {shlex.quote(LOEO_DIR)} {args} "
   f"2>&1 | tee {shlex.quote(LOG)}", cwd=REPO_DIR, env=ENV)
sh(f"{PY} -m floodvalue.model.unet.loeo --table-from {shlex.quote(LOEO_DIR)}", cwd=REPO_DIR, env=ENV)
print(f"LOEO: {time.time() - LOEO_STARTED:.0f} с")


## 5. Выгрузка результатов

Архив `loeo_<MODE>_<commit>.zip` с CSV/JSON фолдов и логом — в `MyDrive/floodvalue/` (в Colab,
`SAVE_TO_DRIVE = "1"`) или скачиванием. Commit клона — в имени архива и в `loeo-summary.json`
(поле `commit`); его же указывают в `docs/verification/transfer-loeo.md`.


In [ ]:
files = sorted(f for f in os.listdir(LOEO_DIR) if f.startswith("loeo"))
assert "loeo-metrics.csv" in files and "loeo-summary.json" in files, files
with open(os.path.join(LOEO_DIR, "COMMIT"), "w") as f:
    f.write(COMMIT + "\n")
archive = shutil.make_archive(os.path.join(WORKDIR, f"loeo_{MODE}_{COMMIT[:12]}"), "zip", LOEO_DIR)
print("архив:", archive, os.path.getsize(archive), "байт", files)

if IN_COLAB:
    if SAVE_TO_DRIVE == "1":
        from google.colab import drive
        drive.mount("/content/drive")
        dest = "/content/drive/MyDrive/floodvalue"
        os.makedirs(dest, exist_ok=True)
        shutil.copy(archive, dest)
        print("скопировано в", dest)
    else:
        from google.colab import files as colab_files
        colab_files.download(archive)
